In [9]:
import pandas as pd
import sqlite3

df = pd.read_csv('C:\\Users\\piyush.c\\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\\Desktop\\Task1\\assignment\\Week3 Sql\\DDS_data_set(Consumables).csv')
df['listPrice'] = pd.to_numeric(df['listPrice'], errors='coerce').fillna(0)
df['make'] = df['partNumber'].str[:2].str.upper() 

conn = sqlite3.connect('dds.db')  
df.to_sql('consumables', conn, if_exists='replace', index=False)  

devices = df[['guid', 'partNumber']].drop_duplicates().rename(columns={'partNumber': 'deviceType'})
devices.to_sql('devices', conn, if_exists='replace', index=False)


conn.close()


C:\Users\piyush.c\AppData\Local\Temp\ipykernel_45720\1084284243.py:4: DtypeWarning: Columns (0: blackLifePages, 1: streetPrice) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('C:\\Users\\piyush.c\\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\\Desktop\\Task1\\assignment\\Week3 Sql\\DDS_data_set(Consumables).csv')


In [10]:
conn = sqlite3.connect('dds.db')
result = pd.read_sql("""
    SELECT make, COUNT(DISTINCT partNumber) as toner_count 
    FROM consumables 
    WHERE tcoCategory LIKE '%TCO%' 
    GROUP BY make 
    ORDER BY toner_count DESC
""", conn)
print(result.head(10))


  make  toner_count
0   10          599
1   TN          400
2   40          366
3   TK          340
4   MX          204
5   CL          195
6   44          191
7   00          182
8   45          163
9   LC          162


In [11]:
result1 = pd.read_sql("""
    SELECT 
        SUBSTR(guid, -4) as region,
        MAX(listPrice) as max_price,
        MIN(listPrice) as min_price
    FROM consumables 
    WHERE tcoCategory = 'TCONormalYield' AND listPrice > 0
    GROUP BY region
""", conn)
print(result1)


Empty DataFrame
Columns: [region, max_price, min_price]
Index: []


In [12]:
result = pd.read_sql("""
    SELECT 
        SUBSTR(guid, -4) as region,
        MAX(listPrice) as max_price,
        MIN(listPrice) as min_price
    FROM consumables 
    WHERE tcoCategory = 'TCONormalYield' AND listPrice > 0
    GROUP BY region
""", conn)
print(result)


Empty DataFrame
Columns: [region, max_price, min_price]
Index: []


In [13]:
result = pd.read_sql("""
    SELECT d.deviceType, GROUP_CONCAT(c.guid) as high_price_guids, COUNT(*) as count
    FROM consumables c
    JOIN devices d ON c.guid = d.guid
    WHERE c.listPrice >= 100
    GROUP BY d.deviceType
    LIMIT 20
""", conn)
print(result)


       deviceType                                   high_price_guids  count
0       001685MIU                                      RC0076,RC0069      2
1       001686MIU                                      RC0076,RC0069      2
2   002028MIU_S36                                             RC0568      1
3       003329MIU  RC0096,RC0096,RC0096,RC0096,RC0074,RC0074,RC00...      8
4       003340MIU  RC0096,RC0096,RC0096,RC0096,RC0074,RC0074,RC00...      8
5       003341MIU  RC0096,RC0096,RC0096,RC0096,RC0074,RC0074,RC00...      8
6       003342MIU  RC0096,RC0096,RC0096,RC0096,RC0074,RC0074,RC00...      8
7       006903MIU                                             RC0036      1
8       006904MIU                                             RC0036      1
9       006905MIU                                             RC0036      1
10      006906MIU                                             RC0036      1
11      006908MIU                               RC0031,RC0022,RC0060      3
12      0069